# Water Quality Prediction: Linear Regression Baseline

This Snowflake notebook follows the same data/feature workflow as the provided benchmark notebook, but swaps the model to the one shown in the title.

**Targets:** Total Alkalinity (TA), Electrical Conductance (EC), Dissolved Reactive Phosphorus (DRP)
**Features (same baseline set):** swir22, NDMI, MNDWI, pet
**Metrics:** R² and RMSE (Train + Test)

> Reminder: Do not use Latitude/Longitude directly as model features per challenge guidance.

## 1) Install dependencies (Snowflake container runtime)
If you already ran the benchmark/getting-started notebooks, you may have `requirements.txt` in your notebook files.
Run the cell below once per fresh container session, then restart the kernel if required.

In [ ]:
# If running on Snowflake container runtime, install packages from requirements.txt
# (If requirements.txt is already present in the left Files panel, this will work as-is.)
!pip install uv
!uv pip install -r "../requirements.txt"

## 2) Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

try:
    from IPython.display import display
except Exception:
    display = print
from sklearn.linear_model import LinearRegression

## 3) Load training data
This expects the same CSVs used by the benchmark notebook to be in the notebook Files panel:
- `water_quality_training_dataset.csv`
- `landsat_features_training.csv`
- `terraclimate_features_training.csv`

In [ ]:
Water_Quality_df = pd.read_csv('../water_quality_training_dataset.csv')
landsat_train_features = pd.read_csv('../landsat_features_training.csv')
Terraclimate_df = pd.read_csv('../terraclimate_features_training.csv')

# Ensure NDMI/MNDWI are numeric
for col in ['NDMI', 'MNDWI']:
    if col in landsat_train_features.columns:
        landsat_train_features[col] = landsat_train_features[col].astype(float)

display(Water_Quality_df.head())
display(landsat_train_features.head())
display(Terraclimate_df.head())

## 4) Join datasets + impute missing values
We concatenate columns (dropping duplicates) and median-impute numeric missing values (same strategy as benchmark).

In [ ]:
def combine_three_datasets(dataset1, dataset2, dataset3):
    data = pd.concat([dataset1, dataset2, dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_three_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
wq_data = wq_data.fillna(wq_data.median(numeric_only=True))
display(wq_data.head())

## 5) Select features + targets
We follow the baseline feature set used in the benchmark workflow: `swir22`, `NDMI`, `MNDWI`, `pet`.

In [ ]:
feature_cols = ['swir22', 'NDMI', 'MNDWI', 'pet']
target_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

wq_model_df = wq_data[feature_cols + target_cols].copy()
X = wq_model_df[feature_cols]
y_TA = wq_model_df['Total Alkalinity']
y_EC = wq_model_df['Electrical Conductance']
y_DRP = wq_model_df['Dissolved Reactive Phosphorus']

display(wq_model_df.head())

## 6) Helper functions (split, scale, train, evaluate)
We compute Train and Test metrics for each target to spot overfitting.

In [ ]:
def split_data(X, y, test_size=0.3, random_state=42):
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

def train_model(X_train_scaled, y_train):
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    return model

MODEL_NAME = 'LinearRegression'
def evaluate_model(model, X_scaled, y_true, dataset_name='Test'):
    y_pred = model.predict(X_scaled)
    r2 = float(r2_score(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f'\n{dataset_name} Evaluation:')
    print(f'R²: {r2:.4f}')
    print(f'RMSE: {rmse:.4f}')
    return y_pred, r2, rmse

def run_pipeline(X, y, param_name='Parameter'):
    print('\n' + '='*60)
    print(f'Training LinearRegression for {param_name}')
    print('='*60)

    X_train, X_test, y_train, y_test = split_data(X, y)
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)
    model = train_model(X_train_scaled, y_train)

    _, r2_train, rmse_train = evaluate_model(model, X_train_scaled, y_train, 'Train')
    _, r2_test, rmse_test = evaluate_model(model, X_test_scaled, y_test, 'Test')

    results = {
        'Model': MODEL_NAME,
        'Parameter': param_name,
        'R2_Train': r2_train,
        'RMSE_Train': rmse_train,
        'R2_Test': r2_test,
        'RMSE_Test': rmse_test
    }
    return model, scaler, pd.DataFrame([results])

## 7) Train + evaluate (3 targets)

In [ ]:
model_TA, scaler_TA, results_TA = run_pipeline(X, y_TA, 'Total Alkalinity')
model_EC, scaler_EC, results_EC = run_pipeline(X, y_EC, 'Electrical Conductance')
model_DRP, scaler_DRP, results_DRP = run_pipeline(X, y_DRP, 'Dissolved Reactive Phosphorus')

results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
display(results_summary)
print('\nMean R² (Test) across 3 targets:', results_summary['R2_Test'].mean())

## 8) Optional: Create submission.csv using validation features
This mirrors the benchmark submission flow. Requires these files in the notebook Files panel:
- `submission_template.csv`
- `landsat_features_validation.csv`
- `terraclimate_features_validation.csv`

In [ ]:
test_file = pd.read_csv('../submission_template.csv')
landsat_val_features = pd.read_csv('../landsat_features_validation.csv')
terraclimate_val_df = pd.read_csv('../terraclimate_features_validation.csv')

for col in ['NDMI', 'MNDWI']:
    if col in landsat_val_features.columns:
        landsat_val_features[col] = landsat_val_features[col].astype(float)

val_data = pd.DataFrame({
    'swir22': landsat_val_features['swir22'].values,
    'NDMI': landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    'pet': terraclimate_val_df['pet'].values,
})
val_data = val_data.fillna(val_data.median(numeric_only=True))

X_scaled_TA = scaler_TA.transform(val_data)
pred_TA = model_TA.predict(X_scaled_TA)
X_scaled_EC = scaler_EC.transform(val_data)
pred_EC = model_EC.predict(X_scaled_EC)
X_scaled_DRP = scaler_DRP.transform(val_data)
pred_DRP = model_DRP.predict(X_scaled_DRP)

submission_df = pd.DataFrame({
    'Longitude': test_file['Longitude'].values,
    'Latitude': test_file['Latitude'].values,
    'Sample Date': test_file['Sample Date'].values,
    'Total Alkalinity': pred_TA,
    'Electrical Conductance': pred_EC,
    'Dissolved Reactive Phosphorus': pred_DRP
})
display(submission_df.head())
submission_df.to_csv('/tmp/linear_regression_submission.csv', index=False)
print('Wrote /tmp/linear_regression_submission.csv')

### Optional: Upload submission file in Snowflake
Uncomment and run if you want to PUT the file into the notebook stage (same pattern as the benchmark notebook).

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql(f"""
PUT file:///tmp/linear_regression_submission.csv
'snow://workspace/USER$.PUBLIC."snowflake-challenge"/versions/live/submission'
AUTO_COMPRESS=FALSE
OVERWRITE=TRUE
""").collect()
print('File saved! Refresh the browser to see it in the sidebar')